# Fake Text-Completion Models

The `fake.py` module provides deterministic text-completion language models for testing.

`FakeListLLM` returns configured string responses in cyclic order. `FakeStreamingListLLM` extends it with synchronous and asynchronous character streaming, optional delays between chunks, and configurable injected failures.

# FakeListLLM

`FakeListLLM` is a fake text-completion model that returns responses from a configured list in order.

After returning the final response, its internal index resets to zero so later invocations begin again with the first response.

## Bases

- `LLM`

## Attributes

1. `responses`: Stores the string responses returned by successive model invocations.

   Responses are selected in order and repeated from the beginning after the last entry.

   * **Type:**
     ```python
     responses: list[str]
     ```

2. `sleep`: Stores an optional delay in seconds for subclasses.

   `FakeListLLM` itself does not use this value. `FakeStreamingListLLM` uses it between streamed characters.

   * **Type:**
     ```python
     sleep: float | None = None
     ```

3. `i`: Stores the index of the response returned by the next invocation.

   The index advances after each synchronous or asynchronous call and resets to zero after the final configured response.

   * **Type:**
     ```python
     i: int = 0
     ```

### Properties

1. `_llm_type`: Returns the identifying model type.
   * **Type:**
     ```python
     _llm_type: str
     ```

   * **Value:**
     ```python
     "fake-list"
     ```

2. `_identifying_params`: Returns the configured responses as the model's identifying parameters.
   * **Type:**
     ```python
     _identifying_params: Mapping[
         str,
         Any
     ]
     ```

   * **Value:**
     ```python
     {
         "responses": self.responses,
     }
     ```

### Methods

1. `_call`: Returns the next configured response synchronously.

   The prompt, stop sequences, callback manager, and additional keyword arguments do not affect response selection. The internal index advances after the response is selected and wraps to zero after the last response.

   * **Syntax:**
     ```python
     _call(
         self,
         prompt: str, # Input prompt
         stop: list[str] | None = None, # Optional stop sequences
         run_manager: CallbackManagerForLLMRun | None = None, # Optional callback manager
         **kwargs: Any # Additional generation parameters
     ) -> str
     ```

2. `_acall`: Returns the next configured response asynchronously.

   No asynchronous external operation is performed. Response selection and index rotation follow the same rules as `_call`.

   * **Syntax:**
     ```python
     async _acall(
         self,
         prompt: str, # Input prompt
         stop: list[str] | None = None, # Optional stop sequences
         run_manager: AsyncCallbackManagerForLLMRun | None = None, # Optional async callback manager
         **kwargs: Any # Additional generation parameters
     ) -> str
     ```

# FakeListLLMError

`FakeListLLMError` is raised by `FakeStreamingListLLM` when streaming reaches the configured failure chunk.

## Bases

- `Exception`

# FakeStreamingListLLM

`FakeStreamingListLLM` extends `FakeListLLM` with character-by-character synchronous and asynchronous streaming.

Each stream first obtains one complete cyclic response through normal invocation and then yields its characters individually.

## Bases

- `FakeListLLM`

## Attributes

1. `error_on_chunk_number`: Stores the zero-based character index at which streaming raises `FakeListLLMError`.

   A value of `None` disables injected failures. The exception is raised before the matching character is yielded.

   * **Type:**
     ```python
     error_on_chunk_number: int | None = None
     ```

### Methods

1. `stream`: Invokes the model synchronously and yields the resulting response one character at a time.

   When `sleep` is configured, `time.sleep` is called before each character. When the current zero-based character index matches `error_on_chunk_number`, `FakeListLLMError` is raised instead of yielding that character.

   The explicit `stop` and additional keyword arguments are accepted by this override but are not forwarded to `invoke`.

   * **Syntax:**
     ```python
     stream(
         self,
         input: LanguageModelInput, # Input accepted by the language model
         config: RunnableConfig | None = None, # Runnable configuration
         *,
         stop: list[str] | None = None, # Optional stop sequences
         **kwargs: Any # Additional streaming parameters
     ) -> Iterator[str]
     ```

2. `astream`: Invokes the model asynchronously and yields the resulting response one character at a time.

   When `sleep` is configured, `asyncio.sleep` is awaited before each character. Injected failure behaviour matches `stream`.

   The explicit `stop` and additional keyword arguments are accepted by this override but are not forwarded to `ainvoke`.

   * **Syntax:**
     ```python
     async astream(
         self,
         input: LanguageModelInput, # Input accepted by the language model
         config: RunnableConfig | None = None, # Runnable configuration
         *,
         stop: list[str] | None = None, # Optional stop sequences
         **kwargs: Any # Additional streaming parameters
     ) -> AsyncIterator[str]
     ```

## Inherited Behaviour

Both fake models inherit the normal `LLM` and Runnable interfaces, including invocation, asynchronous invocation, generation, tracing, batching, binding, retries, and fallback composition.

Inherited methods already documented in the base LLM modules are not repeated here.